# Simulated-data full ST eigen diagnostic

This notebook uses the existing simulated smooth=0.5, nugget=1 data. It first fits the full one-day Vecchia model with the intentionally wrong smoothness `s=1.0`, then reuses that full-data fit summary for dense full-eigen diagnostics on all 25 spatial tiles and all 8 time slots.

Important: the tiles are used only for the full eigen diagnostics. The fitted parameters come from the full one-day Vecchia fit.

In [ ]:
from pathlib import Path
import json
import shlex
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

REPO = Path('/Users/joonwonlee/Documents/GEMS_TCO-1')
FIT_DRIVER = REPO / 'Exercises/st_model/day/amarel_simulation/space_time/eigen_analysis/vecchia_conditional_eigen_sort_sim_smooth0p5_nugget_mismatch_070926.py'
FULL_EIG_SCRIPT = REPO / 'Exercises/st_model/day/amarel_simulation/pure_space/eigen_diagnostic/eig_diag_real_july_st_full_eigen_matern_1of25_8h_070926.py'
assert FIT_DRIVER.exists(), FIT_DRIVER
assert FULL_EIG_SCRIPT.exists(), FULL_EIG_SCRIPT
print('FIT_DRIVER:', FIT_DRIVER)
print('FULL_EIG_SCRIPT:', FULL_EIG_SCRIPT)

## Configure simulation and model

The default local simulated data below exists on this machine and has truth `smooth=0.5`, `nugget=1.0`. The fit model is deliberately `smooth=1.0`, `nugget=1.0`.

In [ ]:
DATA_ROOT = Path('/Users/joonwonlee/Documents/GEMS_DATA/simulation/july_st_circulant_realpattern_smooth0p5')
YEAR = 2023
MONTH = 7
DAY = 1
DAY_IDX = DAY - 1
HOURS_PER_DAY = 8
TRUTH_NUGGET = 1.0

INPUT = DATA_ROOT / f'{YEAR}_july_st_circulant' / f'sim_july{YEAR}_st_circulant_gridded.pkl'
TRUTH_JSON = DATA_ROOT / f'{YEAR}_july_st_circulant' / f'sim_july{YEAR}_st_circulant_truth.json'

# Full-data Vecchia fit: intentionally wrong smoothness, same nugget as DGP.
EXPERIMENT = 'smoothness_mismatch_dgp1'
FIT_MODEL_VARIANT = 'matern_s10_n1_smooth'
FULL_EIG_MODEL_VARIANT = 's10_n1'
RUN_STEM = 'sim_july_st_s05_n0_vecchia_conditional_eigen_sort_smooth_nugget_mismatch_070926'

FIT_OUT_ROOT = REPO / 'outputs/summer_26/local_sim_s05_n1_wrong_s10_vecchia_fit_070926'
FULL_EIG_OUT_ROOT = REPO / 'outputs/summer_26/local_sim_s05_n1_wrong_s10_full_eigen_25tiles_8h_070926'

# 5x5 means 25 separate 1/25 spatial tiles. Fit once, diagnose all 25 tiles.
TILE_GRID = '5x5'
TILE_ROWS = list(range(1, 6))
TILE_COLS = list(range(1, 6))

# Set FORCE_REFIT=True if you want to overwrite/recompute the Vecchia fit artifact.
FORCE_REFIT = False
FORCE_FULL_EIG = False
STOP_ON_TILE_ERROR = True

print('INPUT:', INPUT)
print('TRUTH_JSON:', TRUTH_JSON)
print('FIT_OUT_ROOT:', FIT_OUT_ROOT)
print('FULL_EIG_OUT_ROOT:', FULL_EIG_OUT_ROOT)

## Validate simulated data

In [ ]:
if not INPUT.exists():
    raise FileNotFoundError(f'INPUT does not exist: {INPUT}')
if not TRUTH_JSON.exists():
    raise FileNotFoundError(f'TRUTH_JSON does not exist: {TRUTH_JSON}')

truth = json.loads(TRUTH_JSON.read_text())
print(json.dumps({k: truth.get(k) for k in ['smooth', 'nugget', 'sigmasq', 'range_lat', 'range_lon', 'range_time', 'advec_lat', 'advec_lon', 'n_hours']}, indent=2))
if abs(float(truth['smooth']) - 0.5) > 1e-12:
    raise RuntimeError(f'Expected smooth=0.5 DGP, got {truth["smooth"]}')
if abs(float(truth['nugget']) - TRUTH_NUGGET) > 1e-12:
    raise RuntimeError(f'Expected nugget={TRUTH_NUGGET}, got {truth["nugget"]}')

obj = pd.read_pickle(INPUT)
print('n_hour_keys:', len(obj))
print('first keys:', list(obj.keys())[:8])
first_df = next(iter(obj.values()))
display(first_df.head())
print('first hour shape:', first_df.shape)

## Fit full-data Vecchia model with wrong smoothness s=1.0

This creates a fit artifact containing the estimated covariance parameters. If the artifact already exists and has an `ok` row, it is reused.

In [ ]:
FIT_ARTIFACT = FIT_OUT_ROOT / EXPERIMENT / 'fit_artifacts' / f'{RUN_STEM}_{EXPERIMENT}_{FIT_MODEL_VARIANT}_fit.csv'

def fit_artifact_ok(path):
    if not path.exists():
        return False
    df = pd.read_csv(path)
    return (not df.empty) and ('status' in df.columns) and (df['status'].astype(str).str.lower() == 'ok').any()

print('FIT_ARTIFACT:', FIT_ARTIFACT)
print('artifact ok:', fit_artifact_ok(FIT_ARTIFACT))

In [ ]:
if FORCE_REFIT or not fit_artifact_ok(FIT_ARTIFACT):
    cmd = [
        sys.executable,
        str(FIT_DRIVER),
        '--experiment', EXPERIMENT,
        '--worker-experiment', EXPERIMENT,
        '--worker-model-variant', FIT_MODEL_VARIANT,
        '--worker-stage', 'fit',
        '--data-root', str(DATA_ROOT),
        '--out-root', str(FIT_OUT_ROOT),
        '--years', str(YEAR),
        '--month', str(MONTH),
        '--days', str(DAY_IDX),
        '--hours-per-day', str(HOURS_PER_DAY),
        '--truth-nugget', str(TRUTH_NUGGET),
        '--sim-kind', 'gridded',
        '--device', 'cpu',
        '--cuda-fallback', 'cpu',
        '--target-chunk-size', '16',
        '--diag-chunk-size', '64',
        '--lbfgs-steps', '5',
        '--lbfgs-eval', '20',
        '--lbfgs-history', '10',
        '--grad-tol', '1e-5',
        '--suppress-fit-prints',
    ]
    print(shlex.join(cmd))
    proc = subprocess.Popen(cmd, cwd=str(REPO), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f'Vecchia fit failed with exit code {ret}')
else:
    print('Reusing existing Vecchia fit artifact.')

## Inspect fitted parameters

In [ ]:
fit_df = pd.read_csv(FIT_ARTIFACT)
cols = [
    'status', 'day', 'model_variant', 'model_label', 'smooth', 'vecchia_loss_per_obs',
    'est_sigmasq', 'est_range_lat', 'est_range_lon', 'est_range_time',
    'est_advec_lat', 'est_advec_lon', 'est_nugget', 'fit_s'
]
cols = [c for c in cols if c in fit_df.columns]
display(fit_df[cols])
if not fit_artifact_ok(FIT_ARTIFACT):
    raise RuntimeError('Fit artifact does not contain an ok row.')

## Run all 25 tile full eigen diagnostics using the same full-data fit parameters

In [ ]:
tile_jobs = [(r, c) for r in TILE_ROWS for c in TILE_COLS]
tile_summary_paths = []
tile_curve_paths = []
tile_plot_paths = []
tile_errors = []

for tile_i, (tile_row, tile_col) in enumerate(tile_jobs, start=1):
    out_dir = FULL_EIG_OUT_ROOT / f'year{YEAR}_day{DAY:02d}_tile{tile_row}x{tile_col}_of_{TILE_GRID}'
    summary_csv = out_dir / 'st_full_eigen_summary.csv'
    curves_csv = out_dir / 'st_full_eigen_curves.csv'
    plot_png = out_dir / 'st_full_eigen_model_comparison.png'
    if summary_csv.exists() and curves_csv.exists() and not FORCE_FULL_EIG:
        print(f'[{tile_i:02d}/25] Reusing tile r{tile_row}c{tile_col}: {summary_csv}')
    else:
        cmd = [
            sys.executable,
            str(FULL_EIG_SCRIPT),
            '--input', str(INPUT),
            '--vecchia-summary', str(FIT_ARTIFACT),
            '--output-root', str(FULL_EIG_OUT_ROOT),
            '--year', str(YEAR),
            '--month', str(MONTH),
            '--day', str(DAY),
            '--hours-per-day', str(HOURS_PER_DAY),
            '--tile-grid', TILE_GRID,
            '--tile-row', str(tile_row),
            '--tile-col', str(tile_col),
            '--model-variants', FULL_EIG_MODEL_VARIANT,
        ]
        print(f'[{tile_i:02d}/25] Running tile r{tile_row}c{tile_col}')
        print(shlex.join(cmd))
        proc = subprocess.Popen(cmd, cwd=str(REPO), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='')
        ret = proc.wait()
        if ret != 0:
            msg = f'Full eigen diagnostic failed for tile r{tile_row}c{tile_col} with exit code {ret}'
            tile_errors.append(msg)
            if STOP_ON_TILE_ERROR:
                raise RuntimeError(msg)
            print('WARNING:', msg)
            continue
    tile_summary_paths.append(summary_csv)
    tile_curve_paths.append(curves_csv)
    tile_plot_paths.append(plot_png)

print(f'Done tiles: {len(tile_summary_paths)}/25')
if tile_errors:
    print('\n'.join(tile_errors))

## Aggregate and display all 25 tile outputs

In [ ]:
summary_frames = []
curve_frames = []
for summary_csv, curves_csv in zip(tile_summary_paths, tile_curve_paths):
    s = pd.read_csv(summary_csv)
    c = pd.read_csv(curves_csv)
    for col in ['tile_row', 'tile_col', 'tile_grid', 'n_hours']:
        if col in s.columns and col not in c.columns:
            c[col] = s[col].iloc[0]
    summary_frames.append(s)
    curve_frames.append(c)

all_summary = pd.concat(summary_frames, ignore_index=True)
all_curves = pd.concat(curve_frames, ignore_index=True)
ALL_SUMMARY_CSV = FULL_EIG_OUT_ROOT / 'all_tiles_st_full_eigen_summary.csv'
ALL_CURVES_CSV = FULL_EIG_OUT_ROOT / 'all_tiles_st_full_eigen_curves.csv'
FULL_EIG_OUT_ROOT.mkdir(parents=True, exist_ok=True)
all_summary.to_csv(ALL_SUMMARY_CSV, index=False)
all_curves.to_csv(ALL_CURVES_CSV, index=False)

summary_cols = [
    'tile_row', 'tile_col', 'variant', 'label', 'n_obs', 'loss_per_obs',
    'source_vecchia_loss_per_obs', 'score2_per_df', 'max_abs_bridge',
    'est_sigmasq', 'est_range_lat', 'est_range_lon', 'est_range_time',
    'est_advec_lat', 'est_advec_lon', 'est_nugget'
]
summary_cols = [col for col in summary_cols if col in all_summary.columns]
display(all_summary[summary_cols].sort_values(['tile_row', 'tile_col']))
print('ALL_SUMMARY_CSV:', ALL_SUMMARY_CSV)
print('ALL_CURVES_CSV:', ALL_CURVES_CSV)

# Overview: individual tile curves in gray, interpolated mean curve in model color.
grid = np.linspace(0.0, 1.0, 201)
fig, ax = plt.subplots(figsize=(7.0, 5.6))
for (tile_row, tile_col, variant), g in all_curves.groupby(['tile_row', 'tile_col', 'variant'], sort=True):
    g = g.sort_values('frac_expected')
    ax.plot(g['frac_expected'], g['scaled_cumsum'], color='0.65', alpha=0.35, linewidth=1.0)

for variant, gvar in all_curves.groupby('variant', sort=True):
    ys = []
    for _, g in gvar.groupby(['tile_row', 'tile_col'], sort=True):
        g = g.sort_values('frac_expected')
        ys.append(np.interp(grid, g['frac_expected'], g['scaled_cumsum']))
    mean_y = np.vstack(ys).mean(axis=0)
    label = str(all_summary.loc[all_summary['variant'] == variant, 'label'].iloc[0])
    ax.plot(grid, mean_y, color='#9467bd', linewidth=2.5, label=f'{label} mean over {len(ys)} tiles')

ax.plot([0, 1], [0, 1], color='0.25', linewidth=1.2)
ax.set_xlim(0, 1)
ax.set_ylim(0, max(1.05, float(all_curves['scaled_cumsum'].max()) * 1.03))
ax.grid(True, alpha=0.25)
ax.set_xlabel('projected expected df fraction, sorted by full ST covariance eigenvalue')
ax.set_ylabel('projected cumulative squared score / residual df')
ax.set_title(f'July {YEAR} day={DAY}, all 25 tiles: full ST eigen diagnostic')
ax.legend(loc='best')
fig.tight_layout()
OVERVIEW_PNG = FULL_EIG_OUT_ROOT / 'all_tiles_st_full_eigen_overview.png'
fig.savefig(OVERVIEW_PNG, dpi=220)
plt.close(fig)
display(Image(filename=str(OVERVIEW_PNG)))

# Show the center tile as a concrete example.
center_plot = FULL_EIG_OUT_ROOT / f'year{YEAR}_day{DAY:02d}_tile3x3_of_{TILE_GRID}' / 'st_full_eigen_model_comparison.png'
if center_plot.exists():
    display(Image(filename=str(center_plot)))

## To compare against the true smoothness

Change `FIT_MODEL_VARIANT` to `matern_s05_n1_true` and `FULL_EIG_MODEL_VARIANT` to `s05_n1`, then re-run from the fit cell. The important rule stays the same: fit on the full one-day simulated data once, diagnose separately on the 25 tiles.